# Convert PEFT LoRA Adapters → MLX Format

Converts HuggingFace PEFT adapters to MLX adapter format **without loading the full base model**.
Only the adapter weights (~100MB) are downloaded. RAM usage: <1GB.

Run this once. Output goes to `mlx_adapters/`. Then run `model_comparison_mlx.ipynb`.

In [ ]:
%pip install mlx safetensors huggingface_hub torch -q

In [ ]:
HF_TOKEN = ''

In [ ]:
import os
import json
import numpy as np
import mlx.core as mx
import torch
import safetensors.torch as st
from huggingface_hub import hf_hub_download


def convert_peft_to_mlx_adapter(hf_adapter_repo, output_dir, num_layers,
                                 target_modules, lora_rank, lora_alpha, token=None):
    """
    Convert a HuggingFace PEFT LoRA adapter to MLX adapter format.

    Does NOT load the base model — only downloads and reformats the adapter
    weights (~100MB). RAM usage is negligible.

    Weight mapping:
      PEFT lora_A.weight  (r, in)  -> MLX lora_a  (in, r)  [transposed]
      PEFT lora_B.weight  (out, r) -> MLX lora_b  (r, out) [transposed]

    Key name mapping:
      'base_model.model.model.layers.0.self_attn.q_proj.lora_A.weight'
      -> 'model.layers.0.self_attn.q_proj.lora_a'
    """
    if os.path.exists(output_dir):
        print(f'Skip (exists): {output_dir}')
        return output_dir

    os.makedirs(output_dir, exist_ok=True)
    print(f'Downloading adapter weights from {hf_adapter_repo} ...')

    # Try safetensors first, fall back to bin
    try:
        weights_path = hf_hub_download(
            hf_adapter_repo, 'adapter_model.safetensors', token=token
        )
        peft_weights = st.load_file(weights_path)
    except Exception:
        weights_path = hf_hub_download(
            hf_adapter_repo, 'adapter_model.bin', token=token
        )
        peft_weights = torch.load(weights_path, map_location='cpu')

    print(f'Converting {len(peft_weights)} tensors ...')
    mlx_weights = {}
    skipped = []

    for key, tensor in peft_weights.items():
        # Only process lora_A and lora_B weight matrices
        if not (key.startswith('base_model.model.') and key.endswith('.weight')):
            skipped.append(key)
            continue
        if '.lora_A.' not in key and '.lora_B.' not in key:
            skipped.append(key)
            continue

        # Strip PEFT wrapper prefix and .weight suffix, lowercase lora names
        mlx_key = key[len('base_model.model.') : -len('.weight')]
        mlx_key = mlx_key.replace('.lora_A', '.lora_a').replace('.lora_B', '.lora_b')

        # Transpose: PEFT (r, in) -> MLX (in, r), PEFT (out, r) -> MLX (r, out)
        arr = tensor.float().numpy().T
        mlx_weights[mlx_key] = mx.array(arr)

    if skipped:
        print(f'  Skipped {len(skipped)} non-LoRA keys (embeddings, norms, etc.)')

    # Save adapter weights
    adapter_path = os.path.join(output_dir, 'adapters.safetensors')
    mx.save_safetensors(adapter_path, mlx_weights)
    print(f'  Saved {len(mlx_weights)} tensors -> {adapter_path}')

    # Build MLX adapter_config.json
    # Map PEFT target_module names to layer-relative MLX paths
    attn_modules = {'q_proj', 'k_proj', 'v_proj', 'o_proj'}
    mlp_modules  = {'gate_proj', 'up_proj', 'down_proj'}
    keys = []
    for m in target_modules:
        if m in attn_modules:
            keys.append(f'self_attn.{m}')
        elif m in mlp_modules:
            keys.append(f'mlp.{m}')
        else:
            keys.append(m)

    config = {
        'fine_tune_type': 'lora',
        'num_layers': num_layers,
        'lora_parameters': {
            'rank':    lora_rank,
            'scale':   lora_alpha / lora_rank,
            'dropout': 0.0,
            'keys':    keys,
        },
    }
    config_path = os.path.join(output_dir, 'adapter_config.json')
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    print(f'  Saved config  -> {config_path}')
    print(f'Done: {output_dir}')
    return output_dir

In [ ]:
# Qwen2.5-7B LoRA (28 transformer layers)
convert_peft_to_mlx_adapter(
    hf_adapter_repo='adhamhelmy/qwen2.5-coder-32b-instruct-v1-500',
    output_dir='mlx_adapters/qwen32b_lora',
    num_layers=28,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_rank=32,
    lora_alpha=64,
    token=HF_TOKEN,
)

# Llama-3.1-8B LoRA (32 transformer layers)
# convert_peft_to_mlx_adapter(
#     hf_adapter_repo='adhamhelmy/meta-llama-3.1-8b-instruct-v1-500',
#     output_dir='mlx_adapters/llama8b_lora',
#     num_layers=32,
#     target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
#                     'gate_proj', 'up_proj', 'down_proj'],
#     lora_rank=32,
#     lora_alpha=64,
#     token=HF_TOKEN,
# )

In [ ]:
# Sanity check: confirm files were created and key count looks right
for name, path in [('Qwen7B', 'mlx_adapters/qwen7b_lora'),
                   ('Llama8B', 'mlx_adapters/llama8b_lora'),
                   ('Qwen32B', 'mlx_adapters/qwen32b_lora')]:
    if not os.path.exists(path):
        print(f'{name}: MISSING')
        continue
    files = os.listdir(path)
    weights = mx.load(os.path.join(path, 'adapters.safetensors'))
    print(f'{name}: {len(weights)} weight tensors, files: {files}')
    # Print a sample key to verify naming
    sample_key = next(iter(weights))
    print(f'  sample key: {sample_key}  shape: {weights[sample_key].shape}')